# Customer Churn Prediction

## Phase 3: Data Preprocessing & Feature Engineering

### Objectives
- Separate Features and Target
- Train Validation Test Split
- Feature Engineering
- Encoding Categorical Variables
- Feature Scaling
- Build Preprocessing Pipeline
- Prevent Data Leakage

# Load Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Display first 5 rows
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# Totalcharges Clean

In [2]:
# Remove whitespace
df["TotalCharges"] = df["TotalCharges"].str.strip()

# Convert to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Fill missing values with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Check missing values
print(df["TotalCharges"].isnull().sum())

0


In [3]:
# Separate features and target
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


# Train, Validation and Test Split

In [4]:
# Split training data and test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [5]:
# Split training and validation data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=y_train_val
)

In [6]:
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nChurn distribution in each set:")

print("\nTraining:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTest:")
print(y_test.value_counts(normalize=True) * 100)


Training set: (4929, 19)
Validation set: (1057, 19)
Test set: (1057, 19)

Churn distribution in each set:

Training:
Churn
No     73.463177
Yes    26.536823
Name: proportion, dtype: float64

Validation:
Churn
No     73.415326
Yes    26.584674
Name: proportion, dtype: float64

Test:
Churn
No     73.509934
Yes    26.490066
Name: proportion, dtype: float64


# Feature Engineering

### Create TenureGroup

In [7]:
def add_features(data):
    data = data.copy()

    # 1. Tenure Group
    data["TenureGroup"] = pd.cut(
        data["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=[
            "0-12 Months",
            "13-24 Months",
            "25-48 Months",
            "49-72 Months"
        ]
    )

    # 2. Total Services
    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    data["TotalServices"] = (
        data[service_cols].eq("Yes").sum(axis=1)
        + data["InternetService"].ne("No").astype(int)
    )

    return data

In [8]:
X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
X_test_fe = add_features(X_test)

In [9]:
print("Before Feature Engineering:", X_train.shape)
print("After Feature Engineering:", X_train_fe.shape)

X_train_fe[
    ["tenure", "TenureGroup", "TotalServices"]
].head(10)

Before Feature Engineering: (4929, 19)
After Feature Engineering: (4929, 21)


,tenure,TenureGroup,TotalServices
4926,62,49-72 Months,5
6392,60,49-72 Months,4
6890,53,49-72 Months,4
3194,43,25-48 Months,5
2599,72,49-72 Months,7
4271,28,25-48 Months,1
5384,4,0-12 Months,5
513,56,49-72 Months,7
2577,1,0-12 Months,5
1491,1,0-12 Months,1


### Analyze New Features

In [10]:
print("Tenure Group Distribution:")
print(X_train_fe["TenureGroup"].value_counts().sort_index())

print("\nTotal Services Distribution:")
print(X_train_fe["TotalServices"].value_counts().sort_index())

Tenure Group Distribution:
TenureGroup
0-12 Months     1542
13-24 Months     708
25-48 Months    1099
49-72 Months    1580
Name: count, dtype: int64

Total Services Distribution:
TotalServices
1    881
2    627
3    576
4    656
5    630
6    645
7    482
8    269
9    163
Name: count, dtype: int64


In [11]:
analysis_df = X_train_fe.copy()
analysis_df["Churn"] = y_train

churn_by_tenure = pd.crosstab(
    analysis_df["TenureGroup"],
    analysis_df["Churn"],
    normalize="index"
) * 100

print(churn_by_tenure)

Churn                No        Yes
TenureGroup                       
0-12 Months   53.047990  46.952010
13-24 Months  69.915254  30.084746
25-48 Months  79.890810  20.109190
49-72 Months  90.506329   9.493671


# Categorical Encoding + Numerical Scaling + ColumnTransformer Pipeline

### Import required Libraries

In [12]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

### Numerical and Categorical Columns Identify

In [13]:
# Identify numerical columns
numerical_cols = X_train_fe.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Identify categorical columns
categorical_cols = X_train_fe.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numerical Columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices']

Categorical Columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']


### Create Preprocessing Pipelines

In [14]:
# Numerical preprocessing
numerical_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

### Create ColumnTransformer

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

In [16]:
# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train_fe)

# Transform validation and test data
X_val_processed = preprocessor.transform(X_val_fe)

X_test_processed = preprocessor.transform(X_test_fe)

In [17]:
print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Test shape:", X_test_processed.shape)

print(type(X_train_processed))
print(type(X_val_processed))
print(type(X_test_processed))

Training shape: (4929, 50)
Validation shape: (1057, 50)
Test shape: (1057, 50)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


# Save Preprocessing Pipeline

In [21]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

# Save fitted preprocessor
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully!")


Preprocessor saved successfully!


In [23]:
os.makedirs("../data/processed", exist_ok=True)

joblib.dump(
    {
        "X_train": X_train_processed,
        "X_val": X_val_processed,
        "X_test": X_test_processed,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test
    },
    "../data/processed/processed_data.pkl"
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [24]:
print(os.path.exists("../models/preprocessor.pkl"))
print(os.path.exists("../data/processed/processed_data.pkl"))

True
True
